In [1]:
import sys
import numpy as np
from pathlib import Path

# Add src to python path
# Add project root to python path
project_root = Path.cwd().parent  # notebooks/.. -> project root
sys.path.append(str(project_root))

from src.embeddings.engine import AssetSearchEngine

/home/bigalex95/Projects/Portfolio/Smart-Asset-Search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("--- Starting Experiment ---")

# Initialize Engine
try:
    engine = AssetSearchEngine()
    print(f"Engine initialized on device: {engine.device}")
except Exception as e:
    print(f"Failed to initialize engine: {e}")

--- Starting Experiment ---
Loading model openai/clip-vit-base-patch32 to cuda...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Engine initialized on device: cuda


In [3]:
from src.config import settings

# Image Paths using Config
images_dir = settings.DATA_DIR / "images"
# Using existing files in the directory
dog_img_path = str(images_dir / "dog0.webp")
car_img_path = str(images_dir / "car0.jpeg")

print(f"Dog Image: {dog_img_path}")
print(f"Car Image: {car_img_path}")

Dog Image: /home/bigalex95/Projects/Portfolio/Smart-Asset-Search/data/images/dog0.webp
Car Image: /home/bigalex95/Projects/Portfolio/Smart-Asset-Search/data/images/car0.jpeg


In [4]:
# Get Image Embeddings
print("\nComputing image embeddings...")
dog_vec = np.array(engine.get_image_embedding(dog_img_path))
car_vec = np.array(engine.get_image_embedding(car_img_path))

print(f"Dog vector shape: {dog_vec.shape}")
print(f"Car vector shape: {car_vec.shape}")


Computing image embeddings...
Dog vector shape: (512,)
Car vector shape: (512,)


In [5]:
# Get Text Embedding
print("\nComputing text embedding for 'a dog'...")
text_vec = np.array(engine.get_text_embedding("a dog"))
print(f"Text vector shape: {text_vec.shape}")


Computing text embedding for 'a dog'...
Text vector shape: (512,)


In [6]:
# Calculate Cosine Similarity
# Since vectors are already normalized by the engine, dot product is enough
# Cosine Similarity = (A . B) / (||A|| * ||B||)
# But ||A|| and ||B|| should be 1.0 (checking just in case)

dog_norm = np.linalg.norm(dog_vec)
car_norm = np.linalg.norm(car_vec)
text_norm = np.linalg.norm(text_vec)

print(f"\nVector Norms (should be ~1.0):")
print(f"Dog: {dog_norm:.4f}")
print(f"Car: {car_norm:.4f}")
print(f"Text: {text_norm:.4f}")


Vector Norms (should be ~1.0):
Dog: 1.0000
Car: 1.0000
Text: 1.0000


In [7]:
sim_dog = np.dot(dog_vec, text_vec)
sim_car = np.dot(car_vec, text_vec)

print(f"\n--- Results ---")
print(f"Similarity ('a dog' <-> dog image): {sim_dog:.4f}")
print(f"Similarity ('a dog' <-> car image): {sim_car:.4f}")

if sim_dog > sim_car:
    print("\n✅ SUCCESS: Dog image is more similar to 'a dog' than car image.")
else:
    print("\n❌ FAILURE: Something is wrong. Car image score is higher!")


--- Results ---
Similarity ('a dog' <-> dog image): 0.2674
Similarity ('a dog' <-> car image): 0.1839

✅ SUCCESS: Dog image is more similar to 'a dog' than car image.
